In [1]:
# !pip install reportlab
# !pip install xlsxwriter

<div class="alert alert-info">
    Im ersten Abschnitt werden Inputs und Deklarationen festgelegt, sowie In- und Output-Pfade gesetzt, um diese für spätere Verwendungszwecke erneut verwenden zu können. <br><br>
    Ebenfalls wird das anzustrebende Datenschema aufgezeigt und in einer Liste gespeichert, um dieses jederzeit abzurufen und anpassen zu können
</div>

In [2]:
import pandas as pd
import os
import numpy as np
import folium

from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [3]:
# -------------------------- Basisverzeichnis ----------------------------
base_dir = Path.cwd()

# --------------- Eingabe- und Ausgabeordner definieren ------------------
input_path = base_dir / "Gesammelte_Datenbanken"
output_path = base_dir / "Angepasste_Datenbanken"
data_scheme_path = base_dir / "globales_Datenschema"

print(f"Input-Pfad:  {input_path}")
print(f"Output-Pfad: {output_path}")
print(f"Schema-Pfad: {data_scheme_path}")

Input-Pfad:  C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Gesammelte_Datenbanken
Output-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken
Schema-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\globales_Datenschema


<div class="alert alert-info">
    <b>Zuerst wird eine Liste angelegt mit</b><br>
    - Zielparameter mit Einheit<br>
    - Zielparameter ohne Einheit<br>
    - Einheit des Zielparameters
</div>

In [4]:
# ----------------------- Folgende Attribute werden für die Bachelorarbeit benötigt ------------------
ATTRIBUTES = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L", 
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L", # umol
    "NH4_in_umol/L", # umol
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L", # umol
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L", # umol
    "HS_in_mmol/L",
    "Database_number",
]

# mmol = milli-Mol
# umol = Mikro-Mol


# ---------------------------------- 2. Spalte mit Entfernung aller Suffixe ------------------------------------
SUFFIXES = {
    "_in_mmol/L": "mmol/L",
    "_in_umol/L": "µmol/L",
    "_in_mV": "mV",
    "_in_m": "m",
    "_in_c": "°C",
    "_in_uS/cm": "µS/cm"
}

simplified, units = [], []

for attr in ATTRIBUTES:
    match = next(((s, u) for s, u in SUFFIXES.items() if s in attr), None)
    if match:
        s, u = match
        simplified.append(attr.replace(s, ""))
        units.append(u)
    else:
        simplified.append(attr)
        units.append("")
    
# ---------------------------------- Zusammengefügtes Array mit zwei Spalten -----------------------------------
target_structure = pd.DataFrame({
    "Original": ATTRIBUTES,
    "Simplified": simplified,
    "Unit": units
})


# --------------------------------------------- Anzeigen -------------------------------------------------------
display(target_structure)

,Original,Simplified,Unit
0,location,location,
1,well_or_spring_name,well_or_spring_name,
2,WGS84_latitude,WGS84_latitude,
3,WGS84_longitude,WGS84_longitude,
4,depth_bgl_in_m,depth_bgl,m
5,rock_type,rock_type,
6,stratigraphic_period,stratigraphic_period,
7,temperature_in_c,temperature,°C
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm
9,pH,pH,


<div class="alert alert-info">
    <b>Diese Liste wird ergänzt um</b><br>
    - eine Spalte mit Umrechnungsfaktoren von mg/L in mmol/L oder umol/L<br><br>
    Hierzu werden die Umrechnungsfaktoren
</div>

In [5]:
# ---------------------- Molmassen (g/mol) nach NIST ----------------------
MOLAR_MASS = {
    "O2": 31.9988,
    "Na": 22.989769,
    "Mg": 24.305,
    "Ca": 40.078,
    "Cl": 35.45,
    "SO4": 96.06,
    "HCO3": 61.0168,
    "Li": 6.94,
    "K": 39.0983,
    "Sr": 87.62,
    "NH4": 18.038,
    "Fe": 55.845,
    "Mn": 54.938,
    "F": 18.998,
    "NO3": 62.0049,
    "H2SiO3": 78.11,
    "HS": 33.07
}


# ---------------------- Umrechnungsfaktoren: mg/L -> mmol/L bzw. µmol/L ----------------------

# ---------------------------------- Molmassen als Spalte mappen ------------------------------
target_structure["Molar_Mass_g_per_mol"] = target_structure["Simplified"].map(MOLAR_MASS)


# --------------- Maske: nur Zeilen mit Chemie-Ziel-Einheit und bekannter Molmasse ------------
mask = target_structure["Unit"].isin(["mmol/L", "µmol/L"]) & target_structure["Molar_Mass_g_per_mol"].notna()


# -------------------- Faktor berechnen (mg/L -> mmol/L bzw. µmol/L) --------------------------
# ---------- mg/L / (g/mol) * (1000 mg / 1 g) = mmol/L
# ---------- mg/L / (g/mol) * (1_000_000 mg / 1 g) = µmol/L

target_structure["Conversion_Factor_mg_per_L_to_target"] = np.where(
    mask & target_structure["Unit"].eq("mmol/L"),
    1000 / target_structure["Molar_Mass_g_per_mol"],
    np.where(
        mask & target_structure["Unit"].eq("µmol/L"),
        1_000_000 / target_structure["Molar_Mass_g_per_mol"],
        np.nan
    )
)

# ------------------------------- Ergebnis darstellen -----------------------------------------
display(
    target_structure[["Original", "Simplified", "Unit", "Molar_Mass_g_per_mol", "Conversion_Factor_mg_per_L_to_target"]]
)

,Original,Simplified,Unit,Molar_Mass_g_per_mol,Conversion_Factor_mg_per_L_to_target
0,location,location,,NaN,NaN
1,well_or_spring_name,well_or_spring_name,,NaN,NaN
2,WGS84_latitude,WGS84_latitude,,NaN,NaN
3,WGS84_longitude,WGS84_longitude,,NaN,NaN
4,depth_bgl_in_m,depth_bgl,m,NaN,NaN
5,rock_type,rock_type,,NaN,NaN
6,stratigraphic_period,stratigraphic_period,,NaN,NaN
7,temperature_in_c,temperature,°C,NaN,NaN
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm,NaN,NaN
9,pH,pH,,NaN,NaN


<div class="alert alert-info">
    <b>Erstellen einer .pdf</b><br>
    Diese .pdf dient dazu bei händischer Auswertung der Datenbanken zu vergleichen, welche Attribute in den jeweiligen Dantesätzen vorkommen.
</div>

In [6]:
# ----------------------------- Pfad zu globalem Datenschema -----------------------------
pdf_path = Path(data_scheme_path) / "globales_Datenschema.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)


# --------------------------- Daten für ReportLab vorbereiten ------------------------------
data = [target_structure.columns.tolist()] + target_structure.astype(str).values.tolist()


# ---------------------------------- .pdf erstellen ----------------------------------------
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
tbl = Table(data, repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
]))


# ---------------------------------- .pdf speichern ----------------------------------------
doc.build([tbl])
print(".pdf Pfad:", pdf_path)

.pdf Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\globales_Datenschema\globales_Datenschema.pdf


<div class="alert alert-danger">
    <h1><b>FUNKTIONEN</b></h1>
</div>

In [7]:
# -------------------------------- Hilfsfunktionen --------------------------------
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace("__+", "_", regex=True)
    )
    return df

In [8]:
# ----------------------------- Mappings, die tatsächlich im Datafram enthalten sind -----------------------------
def _available_mapping(df: pd.DataFrame, mapping: dict) -> dict:
    return {src: tgt for src, tgt in mapping.items() if src in df.columns}

In [9]:
# ---------------------------------- Einzelne Tabellenblätter aus Excel-auslesen ---------------------------------
def read_and_map(xls: pd.ExcelFile, sheet_name: str, mapping: dict, usecols=None) -> pd.DataFrame:
    if sheet_name not in xls.sheet_names:
        
        # ----------------------------- leeres DF mit 0 Zeilen
        return pd.DataFrame()
    df = pd.read_excel(xls, sheet_name=sheet_name, usecols=usecols)
    df = _normalize_columns(df)

    # --------------------------------- Fehlerbehandlung Oxygen
    if "dissolved_oxygen_mg_per_l" in df.columns and "dissolved_oxigen_mg_per_l" not in df.columns:
        df = df.rename(columns={"dissolved_oxygen_mg_per_l": "dissolved_oxigen_mg_per_l"})

    ren = _available_mapping(df, mapping)
    if not ren:
        return pd.DataFrame()

    df = df.rename(columns=ren)
    return df[list(ren.values())]

In [10]:
def csv_to_excel(csv_path: Path, excel_path: Path, sheet_name: str = "SHEET1") -> None:

    # ------------------------------ CSV einlesen ---------------------------------
    df = pd.read_csv(csv_path)

    # ------------------------- In Excel schreiben --------------------------------
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)

        # ------------------- Zugriff auf Arbeitsmappe -----------------------------
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        
        # ---------------------- Kopfzeilen-Format --------------------------------
        header_format = workbook.add_format({
            "bold": True,
            "text_wrap": True,
            "valign": "center",
            "fg_color": "#D3D3D3",
            "border": 1
        })
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_format)

        
        # ---------------------- Spaltenbreite dynamisch ---------------------------
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.set_column(i, i, min(max_len, 50))  # begrenze max. Breite

    print(f"Excel-Pfad:\n{excel_path.resolve()}")

In [11]:
# ------------------------------- mg/L zu mmol/L umrechnung ------------------------------
def mgL_to_mmolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # mg/L / (g/mol) = 1e-3 mol/L = mmol/L
    return _clean_numeric(series) / M_gmol

# ------------------------------- mg/L zu umol/L umrechnung ------------------------------
def mgL_to_umolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # (mg/L / g/mol) * 1000 = µmol/L
    return mgL_to_mmolL(series, M_gmol) * 1000.0

In [12]:
# -------------------------- Entfernen von Leerzeichen und N/D Werten ---------------------
def _clean_numeric(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"(?i)\bN/?D\b", "", regex=True)  
         .str.replace(r"[<>≈~]", "", regex=True)       
         .replace({"": np.nan})
         .pipe(pd.to_numeric, errors="coerce")
    )

<div class="alert alert-danger">
    <h1><b>DATENBANK 1BEISPIEL</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.xyz.de <br>
    <b>DATENBANK 1:</b> Komplette BEISPIEL Datenbank, stand DD.MM.YYYY
</div>

<div class="alert alert-info">
    <b>Schritt 0 | Erstellen einer Excel</b><br>
    Da die Datei direkt in einer .csv vorliegt, wird diese zuerst in eine Excel-Datei umgewandelt, um übersichtlicher feststellen zu können, wie das Mapping ablaufen muss
</div>

In [ ]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_AAAAAAAAAAAA_path = Path.cwd() / "Zwischenstände" / "AAAAAAAAAAAA_Datenbank"
print(f"AAAAAAAAAAA-Datenbank: {mid_AAAAAAAAAA_path.resolve()}")

In [ ]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
AAAAAAAAAAAAAA_path = input_path / "AAAAAAAAAAAAA_Geothermal_Springs.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_AAAAAAAAAAAAAA = pd.read_excel(AAAAAAAAAAAAAAAA_path)

# display(df_AAAAAAAAAAAAAAAA.head())

In [ ]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_AAAAAAAAAAAAAAAAA_path / "before_first_transformation_AAAAAAAAAAA.csv"
excel_path = mid_AAAAAAAAAAAAAAAA_path / "after_first_transformation_AAAAAAAAAAAAAA.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

In [ ]:
csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [ ]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "AAAAAAAAAAAAAAAA_SHEET_NAME"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "county": "location",
    "wellname": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "depthlower": "depth_bgl_in_m",
    "lithology": "rock_type",
    "period": "stratigraphic_period",
    "temp": "temperature_in_c",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "li": "Li_in_mmol/L",
    "k": "K_in_mmol/L",
    "sr": "Sr_in_umol/L",
    "nh4": "NH4_in_umol/L",
    "fetot": "Fe_in_mmol/L",
    "mn": "Mn_in_mmol/L",
    "f": "F_in_umol/L",
    "no3": "NO3_in_mmol/L",
    "hs": "HS_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [ ]:
# -------------------------------- Dateien/Ordner --------------------------------
# AAAAAAAAAAAA_path
# AAAAAAAAAAAA_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(AAAAAAAAAAAA_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_AAAAAAAAAAAA_path = mid_AAAAAAAAAAAAA_path / "before_conversion_AAAAAAAAAA_mapped.csv"
df_out.to_csv(out_AAAAAAAAAA_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_AAAAAAAAAAA_path.resolve()}")

<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [ ]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_AAAAAAA_path / "before_conversion_AAAAAAAAAAAA_mapped.csv"
excel_path = mid_AAAAAAAAAAA_path / "before_conversion_AAAAAAAAAAAAA_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "EC_(S_m^-1)" in uS/m</b><br>
    - EC muss von S/m in uS/m umgewandelt werden (ist aktuell in "electrical_conductivity_25c_in_uS/cm" gespeichert)
</div>

In [ ]:
df_out["electrical_conductivity_25c_in_uS/cm"] = pd.to_numeric(df_out["electrical_conductivity_25c_in_uS/cm"], errors="coerce") * 100

In [ ]:
df_out.to_csv(out_AAAAAAAAAAAAA_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [ ]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_AAAAAAAAAAAA_path, index=False, encoding="utf-8-sig")

In [ ]:
csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [ ]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [ ]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_AAAAAAAAAAAA_path = mid_AAAAAAAAAAAAAAAA_path / "after_conversion_AAAAAAAAAAAA_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_AAAAAAAAAA_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_AAAAAAAAAAAA_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_AAAAAAAAAA_path.resolve()}")

<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [ ]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_AAAAAAAAAAAAAA_path / "after_conversion_AAAAAAAAAAAAAAA_mapped.csv"
excel_path = mid_AAAAAAAAAAAAA_path / "after_conversion_AAAAAAAAAAAAAAAAA_mapped_readable.xlsx"

In [ ]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
AAAAAAAAAAAAA_output_path = output_path / "AAAAAAAAAAAAAAAA_cleaned_Database.csv" 
AAAAAAAAAAAAA_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(AAAAAAAAAAAAAA_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {AAAAAAAAAAAAA_output_path.resolve()}")

<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [ ]:
map_path = base_dir / "Kartierung"
AAAAAAAAAAAA_map_path = map_path / "AAAAAAAAAAA_Map.html" 

In [ ]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(AAAAAAAAAAA_map_path)